# ⚡ Execute Commands on Devices with RADKit

![RADKit version](https://img.shields.io/badge/RADKit-1.9.6-blue?logo=cisco&logoColor=white) ![Python version](https://img.shields.io/badge/Python-3.12%2B-purple?logo=python&logoColor=white)

This lab picks up after the RADKit platform overview and focuses on practical command execution workflows.

## What you will learn
- Connect once and reuse the same client session across notebook cells
- Explore and filter inventory to select the right device targets
- Execute operational and configuration commands on one or many devices
- Parse raw CLI output with Genie for automation-friendly structured data
- Download files from devices with resilient SFTP/SCP fallback logic

---

## 1) Load Environment Variables

Start by loading your `.env` file so this notebook can read required values such as:
- `RADKIT_USER`
- `RADKIT_SERVICE`

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

---

## 2) Open a Shared RADKit Session

Next, create one authenticated client session and keep it open for the rest of the notebook.

Using a shared session avoids repeating login logic in every cell and makes examples easier to follow.

In [2]:
from contextlib import ExitStack
from radkit_client.sync import Client

stack = ExitStack()  # This is a Context Manager that allows us to manage the lifecycle of multiple resources (like our Client instance) in a clean and efficient way.
my_client = stack.enter_context(Client.create())  # We create a Client instance and register it

user_id = os.getenv("RADKIT_USER")
my_client.sso_login(user_id)

<frozen radkit_common.utils.ssl>:518: CryptographyDeprecationWarning: Parsed a serial number which wasn't positive (i.e., it was negative or zero), which is disallowed by RFC 5280. Loading this certificate will cause an exception in a future release of cryptography.



A browser window was opened to continue the authentication process. Please follow the instructions there.

Authentication result received.


Client(status='CONNECTED')

---

## 3) Inspect and Filter Inventory

**Why this matters:** Before running automation, confirm which devices are available and choose precise targets.

**Workflow:**
1. Read devices from `service.inventory`.
2. Inspect key attributes such as `name`, `device_type`, and `host`.
3. Filter to a focused subset that matches your use case.

---

### 3.1 List Inventory Details

This example loops through all onboarded devices and prints basic identity fields.

In [4]:
service = my_client.service_cloud(os.getenv("RADKIT_SERVICE")).wait()

print("")
# Going through all the devices in the inventory and printing their parameters
for device in service.inventory.values():
    print(f"📱 Device {device.name} is type {device.device_type} and has host {device.host} ...")


📱 Device iosv-l2 is type IOS_XE and has host 10.10.20.22 ...
📱 Device xrd is type IOS_XR and has host 10.10.20.27 ...
📱 Device ftd-fdm is type FDM and has host 10.10.20.65 ...
📱 Device asav is type ASA and has host 10.10.20.23 ...
📱 Device radkit-service is type RADKIT_SERVICE and has host localhost ...
📱 Device cml is type CML and has host 10.10.20.161 ...
📱 Device c8000v is type IOS_XE and has host 10.10.20.21 ...


---

### 3.2 Filter by Device Type

Use `inventory.filter()` to target only matching devices. This example filters to `IOS_XE` devices before printing names.

In [4]:
# Filtering only IOS_XE devices
for device in service.inventory.filter("device_type", "IOS_XE").values():
    print(f"📱 Device {device.name} is type {device.device_type} ...")

📱 Device c8000v is type IOS_XE ...
📱 Device iosv-l2 is type IOS_XE ...


---

### 3.3 Combine Multiple Filters

You can merge filtered inventories with `|` to build a broader target set. The next example combines `IOS_XR` and `IOS_XE` results into one `DeviceDict`.

In [5]:
# Combining two filtered inventories with | (union) to get both IOS_XR and IOS_XE devices
ios_devices = service.inventory.filter("device_type", "IOS_XR") | service.inventory.filter("device_type", "IOS_XE")
for device in ios_devices.values():
    print(f"📱 Device {device.name} is type {device.device_type} ...")

📱 Device c8000v is type IOS_XE ...
📱 Device iosv-l2 is type IOS_XE ...
📱 Device xrd is type IOS_XR ...


---

## 4) Execute Commands on a Single Device

**Why this matters:** Single-device execution is the safest starting point for validation before wider rollouts.

**Workflow:**
1. Select one `Device` from `service.inventory`.
2. Call `exec()` with a show command or configuration payload.
3. Use `wait()` to get a `SingleExecResponse`.
4. Validate `status` and inspect `data`, `raw_data`, or `errors`.

---

### 4.1 Retrieve Operational Output

This example runs one show command on a target device and inspects the response fields.

In [11]:
from radkit_client import ExecStatus

TARGET_DEVICE = "iosv-l2"

my_device = service.inventory[TARGET_DEVICE]
exec_result = my_device.exec("show ip interface brief").wait()

if exec_result.status == ExecStatus.SUCCESS:
    print(f"\n✅ Execution Status: {exec_result.status}")
    print(f"📟 Device name: {exec_result.device_name}")
    print(f"🧬 Device type: {exec_result.device_type}")
    print(f"🧾 Device command: {exec_result.command}")
    print(f"🆔 Client ID: {exec_result.client_id}")
    print(f"☁️ Service ID: {exec_result.service_id}\n")
    print(f"📦 Raw Data: {exec_result.raw_data}")
else:
    print(f"\n❌ Command execution failed: {str(exec_result.errors)}")
    


✅ Execution Status: ExecStatus.SUCCESS
📟 Device name: iosv-l2
🧬 Device type: IOS_XE
🧾 Device command: show ip interface brief
🆔 Client ID: alfsando@cisco.com
☁️ Service ID: 21km-e0xp-fcib

📦 Raw Data: radkit-iosv#show ip interface brief
Interface              IP-Address      OK? Method Status                Protocol
GigabitEthernet0/0     10.10.20.22     YES TFTP   up                    up      
GigabitEthernet0/1     unassigned      YES unset  up                    up      
GigabitEthernet0/2     unassigned      YES unset  up                    up      
GigabitEthernet0/3     unassigned      YES unset  up                    up      
GigabitEthernet1/0     unassigned      YES unset  up                    up      
Loopback600            unassigned      YES unset  up                    up      
Loopback800            unassigned      YES unset  up                    up      
Loopback801            unassigned      YES unset  up                    up      
Loopback802            unassigned

---

### 4.2 Push Configuration Commands

`exec()` also supports configuration command strings, not only operational reads.

The next example pushes a loopback interface configuration and checks execution status.

Use caution in production environments: validate on lab targets first and prefer idempotent config patterns where possible.

In [10]:
from radkit_client import ExecStatus

TARGET_DEVICE = "iosv-l2"

NEW_CONFIG = f"""configure terminal
interface Loopback802
description TestRADKit
end"""

my_device = service.inventory[TARGET_DEVICE]
exec_result = my_device.exec(NEW_CONFIG).wait()

if exec_result.status == ExecStatus.SUCCESS:
    print(f"\n✅ Execution Status: {exec_result.status}\n")
    print(f"📦 Raw Data: {exec_result.raw_data}")
else:
    print(f"❌ Command execution failed: {str(exec_result.errors)}")


✅ Execution Status: ExecStatus.SUCCESS

📦 Raw Data: radkit-iosv#configure terminal
Enter configuration commands, one per line.  End with CNTL/Z.
radkit-iosv(config)#interface Loopback802
radkit-iosv(config-if)#description TestRADKit
radkit-iosv(config-if)#end
radkit-iosv#


---

## 5) Execute Across Multiple Devices

**Why this matters:** Fan-out workflows let you run one automation step across many devices and aggregate outcomes quickly.

**Workflow:**
1. Build a `DeviceDict` using filters or manual additions.
2. Call `exec()` with one command or a list of commands.
3. Read results by device, by command, or both, depending on response shape.

---

### 5.1 Single Device, Multiple Commands

Pass a list of command strings to `exec()` when one device needs multiple checks in one request.

The response is `ExecResponse_ByCommand_ToSingle`, keyed by command text.

In [14]:
TARGET_DEVICE = "iosv-l2"
QUERY_COMMANDS = ["show version | include Version|uptime", "show memory statistics"]

my_device = service.inventory[TARGET_DEVICE]
exec_result_multiple = my_device.exec(QUERY_COMMANDS).wait()

print(f"\n📦 Output of command `show version | include Version|uptime` is: \n\n{exec_result_multiple['show version | include Version|uptime'].data}\n---------------\n")
print(f"📦 Output of command `show memory statistics` is: \n\n{exec_result_multiple['show memory statistics'].data}\n---------------\n")


📦 Output of command `show version | include Version|uptime` is: 

radkit-iosv#show version | include Version|uptime
Cisco IOS Software, vios_l2 Software (vios_l2-ADVENTERPRISEK9-M), Experimental Version 15.2(20200924:215240) [sweickge-sep24-2020-l2iol-release 135]
radkit-iosv uptime is 1 week, 2 days, 3 hours, 8 minutes
radkit-iosv#
---------------

📦 Output of command `show memory statistics` is: 

radkit-iosv#show memory statistics
                Head    Total(b)     Used(b)     Free(b)   Lowest(b)  Largest(b)
Processor    C2084A0   599981920    59689392   540292528   504464768   501145132
      I/O    79084A0    76546048    63621620    12924428    12920696    12695644
radkit-iosv#
---------------



---

### 5.2 Multiple Devices, Single Command

Run one command against a `DeviceDict` to compare results across devices.

The response type is `ExecResponse_ByDevice_ToSingle`, keyed by device name.

In [15]:
TARGET_DEVICE_1 = "iosv-l2"
TARGET_DEVICE_2 = "c8000v"

QUERY_COMMAND = "show version | include Version|uptime"

# Let's pick our first device by filtering based on the name. This gives us a DeviceDict object
my_devices = service.inventory.filter('name',TARGET_DEVICE_1)

# Next, we add our second device to the same variable with the .add() method
# Notice that we can simply pass the name of the device! The method will look it up in the inventory by itself
my_devices.add(TARGET_DEVICE_2)

# We then execute the same command on both devices at the same time and store the results in a variable
response = my_devices.exec(QUERY_COMMAND).wait()

print("")
for device_response in response.items():
    # The resulting items are tuples where the first element is the device name, and the second is the SingleExecResponse object with all the details and data of the execution for that specific device
    print(f"📱 Device {device_response[0]} : (Status is {device_response[1].status}) Command '{device_response[1].command}' output is:\n{device_response[1].data}\n----------\n")


📱 Device c8000v : (Status is ExecStatus.SUCCESS) Command 'show version | include Version|uptime' output is:
radkit-cat8000v#show version | include Version|uptime
Cisco IOS XE Software, Version 17.15.01a
Cisco IOS Software [IOSXE], Virtual XE Software (X86_64_LINUX_IOSD-UNIVERSALK9-M), Version 17.15.1a, RELEASE SOFTWARE (fc1)
licensed under the GNU General Public License ("GPL") Version 2.0.  The
software code licensed under GPL Version 2.0 is free software that comes
GPL code under the terms of GPL Version 2.0.  For more details, see the
radkit-cat8000v uptime is 1 week, 2 days, 2 hours, 44 minutes
radkit-cat8000v#
----------

📱 Device iosv-l2 : (Status is ExecStatus.SUCCESS) Command 'show version | include Version|uptime' output is:
radkit-iosv#show version | include Version|uptime
Cisco IOS Software, vios_l2 Software (vios_l2-ADVENTERPRISEK9-M), Experimental Version 15.2(20200924:215240) [sweickge-sep24-2020-l2iol-release 135]
radkit-iosv uptime is 1 week, 2 days, 2 hours, 31 minute

---

### 5.3 Multiple Devices, Multiple Commands

You can also pass a command list to a `DeviceDict` for full matrix execution.

The response becomes `ExecResponse_ByDevice_ByCommand`, a nested structure indexed by device, then command.

In [16]:
TARGET_DEVICE_1 = "iosv-l2"
TARGET_DEVICE_2 = "c8000v"

# Our two devices of interest in a DeviceDict object
my_devices = service.inventory.filter('name', TARGET_DEVICE_1)
my_devices.add(TARGET_DEVICE_2)

QUERY_COMMANDS = [
    "show version | include Version|uptime",
    "show processes cpu | include one minute",
]
response = my_devices.exec(QUERY_COMMANDS).wait()

for device_name in [TARGET_DEVICE_1, TARGET_DEVICE_2]:
    print(f"\n📱 {device_name}")
    for command in QUERY_COMMANDS:
        result = response[device_name][command]
        print(f"🧾 {command}")
        print(f"Status: {result.status}")
        print(f"{result.data}\n----------")


📱 iosv-l2
🧾 show version | include Version|uptime
Status: ExecStatus.SUCCESS
radkit-iosv#show version | include Version|uptime
Cisco IOS Software, vios_l2 Software (vios_l2-ADVENTERPRISEK9-M), Experimental Version 15.2(20200924:215240) [sweickge-sep24-2020-l2iol-release 135]
radkit-iosv uptime is 1 week, 2 days, 2 hours, 32 minutes
radkit-iosv#
----------
🧾 show processes cpu | include one minute
Status: ExecStatus.SUCCESS
radkit-iosv#show processes cpu | include one minute
CPU utilization for five seconds: 2%/0%; one minute: 0%; five minutes: 0%
radkit-iosv#
----------

📱 c8000v
🧾 show version | include Version|uptime
Status: ExecStatus.SUCCESS
radkit-cat8000v#show version | include Version|uptime
Cisco IOS XE Software, Version 17.15.01a
Cisco IOS Software [IOSXE], Virtual XE Software (X86_64_LINUX_IOSD-UNIVERSALK9-M), Version 17.15.1a, RELEASE SOFTWARE (fc1)
licensed under the GNU General Public License ("GPL") Version 2.0.  The
software code licensed under GPL Version 2.0 is free s

---

## 6) Parse CLI Output with Genie

`radkit_genie` helps convert raw CLI text into structured data (`QDict`) so automation can access fields directly.

> Genie in RADKit supports more than parsing, but this lab focuses on parser-based workflows.

### Why parse raw CLI output?
Raw output is human-readable but difficult to process reliably in code. Parsed output removes most ad-hoc string matching and regex handling.

---

### 6.1 Execute and Parse One Command

After collecting raw command output, call `parse_text()` with command and platform details.

Supported parser coverage is listed in the [Genie parser catalog](https://pubhub.devnetcloud.com/media/genie-feature-browser/docs/#/parsers).

In [7]:
import json
import radkit_genie

TARGET_DEVICE = "iosv-l2"
QUERY_COMMAND = "show version"

my_device = service.inventory[TARGET_DEVICE]
exec_result_raw = my_device.exec(QUERY_COMMAND).wait()
print(f"\n📋Raw command output:\n{exec_result_raw.data}\n--------------\n")

# Parse the output using Genie
exec_result_parsed = radkit_genie.parse_text(exec_result_raw.data, QUERY_COMMAND, os="iosxe")

# # Convert the QDict to a standard dict before pretty-printing it as JSON
pretty_json = json.dumps(dict(exec_result_parsed), indent=2)
print(f"\n🧞‍♂️Parsed command output (friendlier with your code):\n\n{pretty_json}\n")


📋Raw command output:
radkit-iosv#show version
Cisco IOS Software, vios_l2 Software (vios_l2-ADVENTERPRISEK9-M), Experimental Version 15.2(20200924:215240) [sweickge-sep24-2020-l2iol-release 135]
Copyright (c) 1986-2020 by Cisco Systems, Inc.
Compiled Tue 29-Sep-20 11:53 by sweickge
 
 
ROM: Bootstrap program is IOSv
 
radkit-iosv uptime is 1 week, 2 days, 3 hours, 0 minutes
System returned to ROM by reload
System image file is "flash0:/vios_l2-adventerprisek9-m"
Last reload reason: Unknown reason
 
 
 
This product contains cryptographic features and is subject to United
States and local country laws governing import, export, transfer and
use. Delivery of Cisco cryptographic products does not imply
third-party authority to import, export, distribute or use encryption.
Importers, exporters, distributors and users are responsible for
compliance with U.S. and local country laws. By using this product you
agree to comply with applicable laws and regulations. If you are unable
to comply wi

---

### 6.2 Parse Multiple Devices and Commands

**Why this matters:** Batch parsing keeps response handling consistent at scale.

**Workflow:**
1. Execute multiple commands on multiple devices.
2. Parse the combined response with `radkit_genie.parse()`.
3. Iterate by device and command to inspect `status` and structured `data`.

In [18]:
import json
import radkit_genie

TARGET_DEVICE_1 = "iosv-l2"
TARGET_DEVICE_2 = "c8000v"
    
# Our two devices of interest in a DeviceDict object
my_devices = service.inventory.filter('name', TARGET_DEVICE_1)
my_devices.add(TARGET_DEVICE_2)

commands = [
    "show version",
    "show memory statistics",
]
multiple_response = my_devices.exec(commands).wait()
parsed_response = radkit_genie.parse(multiple_response)

for device_name in [TARGET_DEVICE_1, TARGET_DEVICE_2]:
    print(f"\n📱 {device_name}")
    for command in commands:
        result = parsed_response[device_name][command]
        print(f"🧾 {command}")
        print(f"Status: {result.status}")
        print(f"{json.dumps(dict(result.data.items()), indent=2)}\n----------") # We print the items as a list to avoid printing the entire dictionary if it's too large



📱 iosv-l2
🧾 show version
Status: ExecStatus.SUCCESS
{
  "version": {
    "version_short": "15.2",
    "platform": "vios_l2",
    "version": "15.2(20200924:215240)",
    "image_id": "vios_l2-ADVENTERPRISEK9-M",
    "label": "[sweickge-sep24-2020-l2iol-release 135]",
    "os": "IOS",
    "image_type": "developer image",
    "copyright_years": "1986-2020",
    "compiled_date": "Tue 29-Sep-20 11:53",
    "compiled_by": "sweickge",
    "rom": "Bootstrap program is IOSv",
    "hostname": "radkit-iosv",
    "uptime": "1 week, 2 days, 3 hours, 10 minutes",
    "returned_to_rom_by": "reload",
    "system_image": "flash0:/vios_l2-adventerprisek9-m",
    "last_reload_reason": "Unknown reason",
    "chassis": "IOSv",
    "main_mem": "709857",
    "processor_type": "",
    "rtr_type": "IOSv",
    "chassis_sn": "9GYIA0USR87",
    "number_of_intfs": {
      "Virtual Ethernet": "3",
      "Gigabit Ethernet": "5"
    },
    "mem_size": {
      "non-volatile configuration": "256"
    },
    "processor_

---

## 7) Handle Missing Parsers Safely

Not every command/platform combination has a built-in Genie parser. If parsing fails with `ParserNotFound`, verify support in the [official parser list](https://pubhub.devnetcloud.com/media/genie-feature-browser/docs/#/parsers).

If no parser exists, you can build a custom parser and integrate it in RADKit workflows.

---

### 7.1 Fallback Pattern for Unsupported Commands

In [13]:
import radkit_genie
import json
from genie.libs.parser.utils.common import ParserNotFound

TARGET_DEVICE = "xrd"
QUERY_NONSUPPORTED_COMMAND = "show watchdog"

# Run a command on a specific device
my_device = service.inventory[TARGET_DEVICE]
exec_result_raw = my_device.exec(QUERY_NONSUPPORTED_COMMAND).wait()
print(f"\n📋Raw command output:\n{exec_result_raw.data}\n--------------\n")

# Parse the output using Genie
try:
    exec_result_parsed = radkit_genie.parse_text(exec_result_raw.data, QUERY_NONSUPPORTED_COMMAND, "iosxr")
except ParserNotFound:
    print(f"❌ Parser not found for the command '{QUERY_NONSUPPORTED_COMMAND}' on platform 'iosxr' ❌")
    exec_result_parsed = None

if exec_result_parsed:
    # Convert the QDict to a standard dict before pretty-printing as JSON
    pretty_json = json.dumps(dict(exec_result_parsed), indent=2)
    print(f"\n🧞‍♂️Parsed command output (friendlier with your code):\n\n{pretty_json}\n")
else:
    print("ℹ️ Parsing skipped because no parser is available for this command/platform combination.")


📋Raw command output:
RP/0/RP0/CPU0:ios#show watchdog
Sun Jun  7 17:20:37.469 UTC
---- node0_RP0_CPU0 ----
Memory information:
    Physical Memory	: 6144.0   MB
    Free Memory		: 3793.328 MB
    Memory State	:   Normal
RP/0/RP0/CPU0:ios#
--------------

❌ Parser not found for the command 'show watchdog' on platform 'iosxr' ❌
ℹ️ Parsing skipped because no parser is available for this command/platform combination.


---

## 8) Download Files from Devices (SFTP/SCP)

**Why this matters:** File retrieval is a common automation need for backups, logs, and diagnostics.

**Workflow:**
1. Select the target device from inventory.
2. Prepare the file on the device (optional but common).
3. Attempt SFTP download first.
4. Automatically fallback to SCP if needed.
5. Wait for transfer completion and print actionable error hints if all attempts fail.

In [22]:
TARGET_DEVICE = "iosv-l2"
REMOTE_FILE = "flash:my-backup.cfg"
LOCAL_FILE = "my-backup.cfg"

my_device = service.inventory[TARGET_DEVICE]

# First, let's enable the SCP server in our target device:
print(f"🔧 Enabling SCP server on {TARGET_DEVICE} ...\n")

NEW_CONFIG = f"""configure terminal
ip scp server enable
end"""

exec_result = my_device.exec(NEW_CONFIG).wait()

if exec_result.status == ExecStatus.SUCCESS:
    print(f"\n✅ Execution Status: {exec_result.status}\n")
    print(f"📦 Raw Data: {exec_result.raw_data}")
else:
    print(f"❌ Command execution failed: {str(exec_result.errors)}")
    
# Now, let's generate the backup file and download it
print(f"📁 Backing up startup-config to {REMOTE_FILE} ...\n")

# Try to suppress interactive copy prompts on IOS XE so the workflow stays deterministic.
quiet_prompt_enabled = False
try:
    my_device.exec("configure terminal\nfile prompt quiet\nend").wait()
    quiet_prompt_enabled = True
except Exception as exc:
    print(f"⚠️ Could not enable 'file prompt quiet' (continuing anyway): {exc}")

try:
    copy_output = my_device.exec(f"copy startup-config {REMOTE_FILE}").wait().data
    print(copy_output)

    print(f"\n📁 Verifying the file exists on the device with `dir {REMOTE_FILE}`...\n")
    print(my_device.exec(f"dir {REMOTE_FILE}").wait().data)
    print("\n")

    # Trying SFTP first, then SCP if it fails.
    transfer_ok = False
    transfer_errors = []

    for protocol in ("sftp", "scp"):
        try:
            if protocol == "sftp":
                req = my_device.sftp_download_to_file(remote_path=REMOTE_FILE, local_path=LOCAL_FILE)
            else:
                req = my_device.scp_download_to_file(remote_path=REMOTE_FILE, local_path=LOCAL_FILE)

            # A progress bar will be shown.
            req.show_progress()
            # Wait until the transfer session is closed.
            req.wait_closed()

            print(f"✅ Download completed with {protocol.upper()} from '{REMOTE_FILE}' -> '{LOCAL_FILE}'")
            transfer_ok = True
            break
        except Exception as exc:
            error_text = str(exc)
            transfer_errors.append((protocol, error_text))
            print(f"❌ {protocol.upper()} failed for '{REMOTE_FILE}': {error_text}")

    if not transfer_ok:
        hint_lines = []
        for protocol, error_text in transfer_errors:
            lowered = error_text.lower()
            if protocol == "scp" and "administratively disabled" in lowered:
                hint_lines.append("- SCP appears disabled on the device. Enable it with `ip scp server enable`.")
            if protocol == "sftp" and "0 bytes read on a total of 4 expected bytes" in lowered:
                hint_lines.append("- SFTP handshake failed. Verify SFTP subsystem/support on the device and access policy path.")

        if not hint_lines:
            hint_lines.append("- Verify SSH reachability, credentials/AAA permissions, and device file-transfer protocol support.")

        hint_text = "\n".join(dict.fromkeys(hint_lines))
        all_errors = "\n".join([f"- {proto.upper()}: {msg}" for proto, msg in transfer_errors])

        raise RuntimeError(
            "❌ Download failed with all protocol attempts.\n"
            f"Errors:\n{all_errors}\n"
            f"Suggested fixes:\n{hint_text}"
        )
finally:
    if quiet_prompt_enabled:
        try:
            my_device.exec("configure terminal\nno file prompt quiet\nend").wait()
        except Exception as exc:
            print(f"⚠️ Could not restore prompt behavior ('no file prompt quiet'): {exc}")

🔧 Enabling SCP server on iosv-l2 ...


✅ Execution Status: ExecStatus.SUCCESS

📦 Raw Data: radkit-iosv#configure terminal
Enter configuration commands, one per line.  End with CNTL/Z.
radkit-iosv(config)#ip scp server enable
radkit-iosv(config)#end
radkit-iosv#
📁 Backing up startup-config to flash:my-backup.cfg ...

radkit-iosv#copy startup-config flash:my-backup.cfg
3233 bytes copied in 0.407 secs (7943 bytes/sec)
radkit-iosv#

📁 Verifying the file exists on the device with `dir flash:my-backup.cfg`...

radkit-iosv#dir flash:my-backup.cfg
Directory of flash0:/my-backup.cfg
 
  271  -rw-        3233   Jun 7 2026 17:16:26 +00:00  my-backup.cfg
 
2142715904 bytes total (2017714176 bytes free)
radkit-iosv#


❌ SFTP failed for 'flash:my-backup.cfg': Performing action failed: 0 bytes read on a total of 4 expected bytes
my-backup.cfg   0.0% [>                               ]   0/3233  eta [?:??:??]






 my-backup.cfg 100.0% [=================================>] 3233/3233 eta [00:00]
✅ Downl

---

You have completed the command execution lab.

Run the final cell to close the shared client session and release resources cleanly.

In [ ]:
stack.close()